In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime

# Create a Partitioned Table

In [0]:
%sql
DROP TABLE IF EXISTS oc_catalog.oc_schema.snapshot_table;

CREATE TABLE IF NOT EXISTS oc_catalog.oc_schema.snapshot_table
(
    snapshot_date DATE,
    first_name STRING,
    last_name STRING,
    country STRING
)
PARTITIONED BY (snapshot_date)

# Fetch the Last Date of the Last Month

In [0]:
last_date_of_last_month = spark.sql("SELECT last_day(add_months(current_date(), -1))").collect()[0][0]
print(last_date_of_last_month)
print(type(last_date_of_last_month)) # It is a Date Type Variable

# Convert a String to DateTime
* It is possible to convert a <b>string</b> to a <b>datetime</b> using the <b>Python</b> function <b>datetime.strptime ()</b>.
* This function takes two arguments -
  * First argument is the <b>string</b> which needs to be converted to <b>datetime</b>.
  * Second argument is the format of the <b>datetime</b>, like '<b>YYYY-MM-dd hh:mm:ss</b>'.

In [0]:
print(datetime.strptime("2026-02-04", '%Y-%m-%d'))
print(type(datetime.strptime("2026-02-04", '%Y-%m-%d')))

# Convert a String to Date
* It is also possible to convert a <b>string</b> to a <b>date</b> using the <b>Python</b> function <b>datetime.strptime ()</b>.
* After converting a <b>string</b> to a <b>datetime</b> using the <b>Python</b> function <b>datetime.strptime ()</b>, get only the <b>date</b> part from the <b>datetime</b>, by adding the function <b>.date ()</b>.

In [0]:
print(datetime.strptime("2026-02-04", '%Y-%m-%d').date())
print(type(datetime.strptime("2026-02-04", '%Y-%m-%d').date()))

# Create Dataset

In [0]:
# List of Data
snapshot_data = [
    (datetime.strptime("2025-09-30", '%Y-%m-%d').date(), "Oindrila", "Chakraborty", "Canada"),
    (datetime.strptime("2025-09-30", '%Y-%m-%d').date(), "Soumyajyoti", "Bagchi", "Mexico"),
    (datetime.strptime("2025-10-31", '%Y-%m-%d').date(), "Oindrila", "Chakraborty", "Argentina"),
    (datetime.strptime("2025-10-31", '%Y-%m-%d').date(), "Soumyajyoti", "Bagchi", "Cuba"),
    (datetime.strptime("2025-11-30", '%Y-%m-%d').date(), "Oindrila", "Chakraborty", "Georgia"),
    (datetime.strptime("2025-12-31", '%Y-%m-%d').date(), "Soumyajyoti", "Bagchi", "Germany")
]

# Define the Schema Explicitly
snapshot_schema = StructType([
    StructField("snapshot_date", DateType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("country", StringType(), True)
])

# Create DataFrame
df_snapshot = spark.createDataFrame(snapshot_data, snapshot_schema)

# Show the DataFrame and its Schema
df_snapshot.display()
df_snapshot.printSchema()

# Insert Data from Dataset into Partitioned Table Using PySpark for the First Time

In [0]:
df_snapshot\
        .write\
        .partitionBy("snapshot_date")\
        .mode("append")\
        .saveAsTable("oc_catalog.oc_schema.snapshot_table")

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.snapshot_table;

# Create Source View to Append Data into a Partitioned Table

In [0]:
# List of Data
snapshot_data = [
    (datetime.strptime("2026-01-31", '%Y-%m-%d').date(), "Oindrila", "Chakraborty", "India"),
    (datetime.strptime("2026-01-31", '%Y-%m-%d').date(), "Soumyajyoti", "Bagchi", "India")
]

# Define the Schema Explicitly
snapshot_schema = StructType([
    StructField("snapshot_date", DateType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("country", StringType(), True)
])

# Create DataFrame
df_snapshot = spark.createDataFrame(snapshot_data, snapshot_schema)

# Show the DataFrame and its Schema
df_snapshot.display()
df_snapshot.printSchema()

# Create Source View
df_snapshot.createOrReplaceTempView("vw_snapshot")

# Append Data from Source View into the Existing Partitioned Table Using Spark SQL

In [0]:
spark.sql(f"""
INSERT INTO oc_catalog.oc_schema.snapshot_table
PARTITION (snapshot_date = '{last_date_of_last_month}')
SELECT * EXCEPT (snapshot_date)
FROM vw_snapshot
""")

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.snapshot_table;

# Create Dataset to Replace a Particular Partition

In [0]:
# List of Data
snapshot_data = [
    (datetime.strptime("2026-01-31", '%Y-%m-%d').date(), "Oindrila", "Chakraborty", "South Korea"),
    (datetime.strptime("2026-01-31", '%Y-%m-%d').date(), "Soumyajyoti", "Bagchi", "Belgium")
]

# Define the Schema Explicitly
snapshot_schema = StructType([
    StructField("snapshot_date", DateType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("country", StringType(), True)
])

# Create DataFrame
df_snapshot = spark.createDataFrame(snapshot_data, snapshot_schema)

# Show the DataFrame and its Schema
df_snapshot.display()
df_snapshot.printSchema()

# Overwrite Records for the Partition Where Snapshot Date is Equal to Last Date of Last Month Using PySpark DataFrame

In [0]:
df_snapshot\
        .write\
        .mode("overwrite")\
        .option("replaceWhere", f"snapshot_date = '{last_date_of_last_month}'")\
        .saveAsTable("oc_catalog.oc_schema.snapshot_table")

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.snapshot_table;

# Create Source View to Replace a Particular Partition

In [0]:
# List of Data
snapshot_data = [
    (datetime.strptime("2026-01-31", '%Y-%m-%d').date(), "Oindrila", "Chakraborty", "Singapore"),
    (datetime.strptime("2026-01-31", '%Y-%m-%d').date(), "Soumyajyoti", "Bagchi", "Sri Lanka")
]

# Define the Schema Explicitly
snapshot_schema = StructType([
    StructField("snapshot_date", DateType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("country", StringType(), True)
])

# Create DataFrame
df_snapshot = spark.createDataFrame(snapshot_data, snapshot_schema)

# Show the DataFrame and its Schema
df_snapshot.display()
df_snapshot.printSchema()

# Create Source View
df_snapshot.createOrReplaceTempView("vw_snapshot")

# Overwrite Records for the Partitions Using Dynamic Partition Overwrites
* <b>Dynamic Partition Overwrite</b> updates only the <b>partitions</b> for which the new data contains records, leaving other <b>partitions</b> unchanged.
* It is not required to specify the <b>exact partition values</b> in the <b>write</b> command itself. <b>Spark</b> automatically determines which <b>partitions</b> to overwrite based on the data being written.
* <b>Databricks Runtime 16.3</b> and above supports <b>Dynamic Partition Overwrites</b> for <b>partitioned tables</b> using <b>REPLACE USING</b> clause.
* <b>REPLACE USING</b> clause only overwrites the <b>partitions</b> targeted by the incoming data. All other <b>partitions</b> remain unchanged.

In [0]:
%sql
INSERT INTO TABLE oc_catalog.oc_schema.snapshot_table
    REPLACE USING (snapshot_date)
    SELECT * FROM vw_snapshot

In [0]:
%sql
SELECT * FROM oc_catalog.oc_schema.snapshot_table;